<a href="https://colab.research.google.com/github/Di-oss/com4/blob/%D0%90%D0%B1%D1%80%D0%B0%D0%BC%D0%BE%D0%B2%D0%B0/dev_abramova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Модуль 3: Управление заказами
# Интернет-магазин "Все и сразу"

from datetime import datetime, timedelta
import uuid

class OrderItem:
    """Позиция заказа"""
    def __init__(self, product_id, product_name, price, quantity):
        self.id = str(uuid.uuid4())[:8]
        self.product_id = product_id
        self.product_name = product_name
        self.price = price
        self.quantity = quantity
        self.discount = 0

    def get_total(self):
        return self.price * self.quantity * (1 - self.discount/100)

class Order:
    """Класс заказа"""
    _counter = 1000

    def __init__(self, customer_id, customer_name):
        Order._counter += 1
        self.id = str(uuid.uuid4())[:8]
        self.order_number = f"ORD-{Order._counter}"
        self.customer_id = customer_id
        self.customer_name = customer_name
        self.order_date = datetime.now()
        self.status = "Новый"
        self.items = []
        self.total_amount = 0
        self.delivery_address = ""
        self.delivery_method = ""
        self.delivery_cost = 0
        self.payment_method = ""
        self.payment_status = "Не оплачен"
        self.tracking_number = ""

    def add_item(self, product_id, product_name, price, quantity):
        item = OrderItem(product_id, product_name, price, quantity)
        self.items.append(item)
        self.calculate_total()
        return item

    def remove_item(self, item_id):
        for i, item in enumerate(self.items):
            if item.id == item_id:
                removed = self.items.pop(i)
                self.calculate_total()
                print(f"Товар {removed.product_name} удален")
                return True
        return False

    def calculate_total(self):
        self.total_amount = sum(item.get_total() for item in self.items) + self.delivery_cost

    def set_delivery(self, method, address, cost=0):
        self.delivery_method = method
        self.delivery_address = address
        self.delivery_cost = cost
        self.calculate_total()
        print(f"Доставка: {method}, адрес: {address}")

    def set_payment(self, method):
        self.payment_method = method
        print(f"Оплата: {method}")

    def process_payment(self):
        if self.payment_method == "Карта":
            self.payment_status = "Оплачен"
            self.status = "Подтвержден"
            print("Оплата прошла успешно")
            return True
        elif self.payment_method == "Наличные":
            self.payment_status = "Ожидает оплаты"
            self.status = "Оформлен"
            print("Заказ будет оплачен при получении")
            return True
        return False

    def update_status(self, new_status):
        old = self.status
        self.status = new_status
        print(f"Статус изменен: {old} -> {new_status}")
        if new_status == "Отправлен":
            self.tracking_number = f"TRACK-{uuid.uuid4().hex[:6].upper()}"

    def cancel(self):
        if self.status in ["Новый", "Оформлен", "Подтвержден"]:
            self.status = "Отменен"
            print("Заказ отменен")
            return True
        print("Заказ нельзя отменить")
        return False

    def get_info(self):
        info = f"\n=== ЗАКАЗ {self.order_number} ===\n"
        info += f"Клиент: {self.customer_name}\n"
        info += f"Дата: {self.order_date.strftime('%d.%m.%Y %H:%M')}\n"
        info += f"Статус: {self.status}\n"
        info += f"Оплата: {self.payment_status}\n"
        info += "--- Товары ---\n"
        for i, item in enumerate(self.items, 1):
            info += f"{i}. {item.product_name} x{item.quantity} = {item.get_total():.2f} руб.\n"
        info += f"Доставка: {self.delivery_cost:.2f} руб.\n"
        info += f"ИТОГО: {self.total_amount:.2f} руб.\n"
        if self.tracking_number:
            info += f"Трек: {self.tracking_number}\n"
        info += "="*30
        return info

class OrderManager:
    """Менеджер заказов"""
    def __init__(self):
        self.orders = {}  # id -> Order
        self.customer_orders = {}  # customer_id -> [order_ids]

    def create_order(self, customer_id, customer_name):
        order = Order(customer_id, customer_name)
        self.orders[order.id] = order

        if customer_id not in self.customer_orders:
            self.customer_orders[customer_id] = []
        self.customer_orders[customer_id].append(order.id)

        print(f"Заказ {order.order_number} создан")
        return order

    def get_order(self, order_id):
        return self.orders.get(order_id)

    def get_order_by_number(self, order_number):
        for order in self.orders.values():
            if order.order_number == order_number:
                return order
        return None

    def get_customer_orders(self, customer_id):
        order_ids = self.customer_orders.get(customer_id, [])
        return [self.orders[oid] for oid in order_ids if oid in self.orders]

    def get_orders_by_status(self, status):
        return [o for o in self.orders.values() if o.status == status]

    def list_all_orders(self):
        if not self.orders:
            print("Нет заказов")
            return
        print("\n=== ВСЕ ЗАКАЗЫ ===")
        for order in self.orders.values():
            print(f"{order.order_number} | {order.customer_name} | {order.status} | {order.total_amount:.2f} руб.")
        print("="*30)

# Тестовые данные
manager = OrderManager()

# Создаем тестового пользователя (для связи с модулем 4)
test_customer_id = "cust_123"
test_customer_name = "Иван Петров"

# Главный цикл
while True:
    print("\n" + "="*50)
    print("МОДУЛЬ УПРАВЛЕНИЯ ЗАКАЗАМИ")
    print("="*50)
    print("1 - Создать новый заказ")
    print("2 - Найти заказ по номеру")
    print("3 - Мои заказы")
    print("4 - Все заказы")
    print("5 - Работа с заказом")
    print("0 - Выход")

    choice = input("\nВыберите действие: ")

    if choice == "1":
        # Создание заказа
        print("\n--- НОВЫЙ ЗАКАЗ ---")
        # В реальности customer_id берется из модуля регистрации
        customer_id = input("ID покупателя (Enter для тестового): ") or test_customer_id
        customer_name = input("Имя покупателя (Enter для тестового): ") or test_customer_name

        order = manager.create_order(customer_id, customer_name)

        # Добавление товаров
        while True:
            print("\nДобавление товара:")
            product_name = input("Название товара (или 'стоп'): ")
            if product_name.lower() == "стоп":
                break

            try:
                price = float(input("Цена: "))
                quantity = int(input("Количество: "))
                order.add_item(len(order.items)+1, product_name, price, quantity)
                print(f"Товар добавлен. Текущая сумма: {order.total_amount:.2f} руб.")
            except ValueError:
                print("Ошибка ввода")

        # Настройка доставки
        print("\n--- ДОСТАВКА ---")
        print("1. Курьер (500 руб.)")
        print("2. Самовывоз (0 руб.)")
        print("3. Почта (300 руб.)")
        del_choice = input("Выберите способ: ")

        address = input("Адрес доставки: ")
        if del_choice == "1":
            order.set_delivery("Курьер", address, 500)
        elif del_choice == "2":
            order.set_delivery("Самовывоз", address, 0)
        elif del_choice == "3":
            order.set_delivery("Почта", address, 300)
        else:
            order.set_delivery("Курьер", address, 500)

        # Настройка оплаты
        print("\n--- ОПЛАТА ---")
        print("1. Карта онлайн")
        print("2. Наличные при получении")
        pay_choice = input("Выберите способ: ")

        if pay_choice == "1":
            order.set_payment("Карта")
        else:
            order.set_payment("Наличные")

        # Подтверждение
        print(order.get_info())
        confirm = input("\nПодтвердить заказ? (да/нет): ")
        if confirm.lower() == "да":
            order.process_payment()
            print("Заказ оформлен!")
        else:
            order.cancel()
            print("Заказ отменен")

    elif choice == "2":
        # Поиск заказа
        order_num = input("Введите номер заказа (например ORD-1001): ")
        order = manager.get_order_by_number(order_num)
        if order:
            print(order.get_info())
        else:
            print("Заказ не найден")

    elif choice == "3":
        # Заказы покупателя
        customer_id = input("Введите ID покупателя (Enter для тестового): ") or test_customer_id
        orders = manager.get_customer_orders(customer_id)
        if orders:
            print(f"\n=== ЗАКАЗЫ ПОКУПАТЕЛЯ {customer_id} ===")
            for order in orders:
                print(f"{order.order_number} | {order.order_date.strftime('%d.%m.%Y')} | {order.status} | {order.total_amount:.2f} руб.")
        else:
            print("У покупателя нет заказов")

    elif choice == "4":
        # Все заказы
        manager.list_all_orders()

    elif choice == "5":
        # Работа с конкретным заказом
        order_num = input("Введите номер заказа: ")
        order = manager.get_order_by_number(order_num)
        if not order:
            print("Заказ не найден")
            continue

        while True:
            print("\n" + "-"*30)
            print(f"ЗАКАЗ {order.order_number}")
            print("-"*30)
            print("1 - Показать информацию")
            print("2 - Изменить статус")
            print("3 - Добавить товар")
            print("4 - Удалить товар")
            print("5 - Отменить заказ")
            print("0 - Назад")

            sub_choice = input("Выберите: ")

            if sub_choice == "1":
                print(order.get_info())
            elif sub_choice == "2":
                print("Статусы: Новый, Подтвержден, Отправлен, Доставлен")
                new_status = input("Новый статус: ")
                order.update_status(new_status)
            elif sub_choice == "3":
                name = input("Название товара: ")
                try:
                    price = float(input("Цена: "))
                    qty = int(input("Количество: "))
                    order.add_item(len(order.items)+1, name, price, qty)
                    print("Товар добавлен")
                except:
                    print("Ошибка ввода")
            elif sub_choice == "4":
                for i, item in enumerate(order.items, 1):
                    print(f"{i}. {item.product_name} x{item.quantity}")
                try:
                    num = int(input("Номер товара для удаления: ")) - 1
                    if 0 <= num < len(order.items):
                        order.remove_item(order.items[num].id)
                except:
                    print("Ошибка ввода")
            elif sub_choice == "5":
                order.cancel()
            elif sub_choice == "0":
                break

    elif choice == "0":
        print("Программа завершена")
        break

    else:
        print("Неверный выбор")